# SimpleAI — TinyGPT Additive Batch Training & Evaluation Notebook
### Sequential Execution + Real-Time Tracking + Zero-Fault Evaluation

> **Protocol**: Sequential batch execution. Runs experiments one by one with memory cleanup, zero kernel crash, and automatic 40-question strict benchmark scorecard generation.


### Step 1: Hardware Diagnostics & Google Drive Setup
Detect available compute runtime (GPU / CPU) and mount Google Drive for continuous artifact persistence.


In [ ]:
import os, sys, time, json, shutil
import torch
from google.colab import drive

print("=" * 65)
print("🚀 SimpleAI Experiment Environment Diagnostics")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 检测成功: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ 未检测到 GPU，将在 CPU 模式下运行（建议切换至 GPU 运行时以提高训练速度）")
print("=" * 65)

# 挂载 Google Drive，完成的 artifact 自动保存到此处
try:
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/SimpleAI_Experiments'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"✅ Google Drive 挂载成功！全部产物将自动归档至: {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = None
    print(f"⚠️ Google Drive 挂载跳过: {e}，产物将保存在 Colab 本地运行区")


### Step 2: Clone Codebase and Configurations
Clone the latest repository containing execution modules, configurations, and evaluation benchmarks.


In [ ]:
# 1. Install core dependencies
!pip install -q torch openpyxl huggingface_hub pandas matplotlib tabulate

import os, sys

# 2. 直连 Hugging Face 公开仓库
os.chdir('/content')
WORKSPACE = "/content/additive-rand-transformer"

if not os.path.exists(WORKSPACE):
    print("🌐 正在从 Hugging Face 极速克隆仓库与 440 项实验配置...")
    os.system(f"git clone https://huggingface.co/Hana-ame/additive-rand-transformer {WORKSPACE}")
else:
    print("🔄 仓库已存在，拉取 Hugging Face 最新代码...")
    os.system(f"git -C {WORKSPACE} pull || true")

os.chdir(WORKSPACE)

# 3. 配置 Python sys.path
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

from additive_rand_transformer.model import TinyGPT, TinyGPTConfig, VOCAB_SIZE, TOK_TO_ID
from additive_rand_transformer.data import BOS, EOS, PLUS, MINUS, EQ, SP, ANS, ANS_END, _int_to_tokens, extract_answer
print(f"✅ 环境准备完毕！当前工作区: {WORKSPACE}")
print(f"✅ 词表大小: {VOCAB_SIZE} Tokens (已启用 <ANS> ... </ANS> 零容错闭合严格判定)")


### Step 3: Zero-Tolerance 40-Benchmark Scoring Engine
Loads the 40-question benchmark suite (`testset_adder.json`) to evaluate exact match arithmetic reasoning across digits 1 to 4.


In [ ]:
# Benchmark scoring module loaded from additive_rand_transformer.batch_train
from additive_rand_transformer.batch_train import run_batch_experiments, evaluate_all_saved_checkpoints
print("Benchmark scoring engine ready.")


### Step 4: Batch Sequential Training & Real-Time Evaluation
Execute target experiment suites sequentially. Evaluates loss convergence, exact match accuracy, and formatting compliance.


In [ ]:
# ==============================================================================
# Execution mode selection
# ==============================================================================
# 可选: "FRONTIER_197_204" (优先机制突破 8 项), "TINY_SCALING" (221-241), "RUN_ALL_UNRUN" (全部未跑)
RUN_MODE = "FRONTIER_197_204"
MAX_EXPERIMENTS = 32       # 本次批量运行的上限个数
SAVE_RUNS_DIR = "runs/notebook"
# ==============================================================================

import os, sys, time, json, random, gc
import torch
from additive_rand_transformer.model import TinyGPT, TinyGPTConfig
from additive_rand_transformer.data import make_single_cot_batch
from additive_rand_transformer.batch_train import (
    select_configs, parse_exp_mechanism, resolve_bias,
    verify_model_40_questions, BLOCK, MAX_DIGITS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 启动 Notebook 顺序批量实测引擎，执行硬件: {device}")

target_configs = select_configs(RUN_MODE, os.path.join(WORKSPACE, "configs"), MAX_EXPERIMENTS)
print(f"📋 共筛选出 {len(target_configs)} 个待运行实验:")
for f in target_configs:
    print(f"  - {f}")

all_experiment_reports = []

for idx, cfg_file in enumerate(target_configs, 1):
    cfg_path = os.path.join(WORKSPACE, "configs", cfg_file)
    with open(cfg_path, "r", encoding="utf-8") as fp:
        cd = json.load(fp)
    
    title = cd.get("test_objective", cd.get("title", cfg_file))
    print("\n" + "=" * 75)
    print(f"▶ [{idx}/{len(target_configs)}] 正在训练实验: {cfg_file}")
    print(f"   标题: {title}")
    print("=" * 75)
    
    mech = parse_exp_mechanism(cfg_file, cd)
    steps = int(cd.get("steps", 4000))
    bs = int(cd.get("batch_size", 32))
    bias = resolve_bias(cd)
    rng = random.Random(int(cd.get("seed", 1337)))
    
    cfg_kwargs = dict(
        vocab_size=mech["vocab_size"], block_size=BLOCK,
        n_layer=int(cd.get("layers", 4)), n_head=mech["heads"],
        n_embd=int(cd.get("d", 128)),
    )
    if mech["looped_ut"]:
        cfg_kwargs.update(looped_ut=True, looped_ut_steps=mech["looped_ut_steps"])
    mcfg = TinyGPTConfig(**cfg_kwargs)
    model = TinyGPT(mcfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
    
    run_dir = os.path.join(SAVE_RUNS_DIR, cfg_file[:-5])
    os.makedirs(run_dir, exist_ok=True)
    
    t0 = time.time()
    model.train()
    last_loss = 0.0
    
    # 动态单步按需生成，零内存堆积
    for step in range(steps):
        lr = 3e-4 * min(1.0, (step + 1) / 200)
        for pg in opt.param_groups:
            pg["lr"] = lr
        
        x, y = make_single_cot_batch(
            rng, BLOCK, bs, device=device,
            max_digits=MAX_DIGITS, four_digit_bias=bias,
            answer_order=mech["answer_order"],
            avalanche=mech["avalanche"],
            carry_depth=0 if mech["carry_curriculum"] else None,
            self_verify=mech["self_verify"],
            use_ans_tags=mech["use_ans_tags"],
        )
        
        logits, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        last_loss = float(loss.item())
        
        if (step + 1) % 500 == 0 or (step + 1) == steps:
            print(f"    Step [{step+1:>5}/{steps}] | Loss: {last_loss:.4f} | LR: {lr:.2e} | 耗时: {time.time()-t0:.1f}s")
            
    train_duration = time.time() - t0
    
    # 持久化完整配置与 Checkpoint
    ckpt_path = os.path.join(run_dir, "checkpoint_final.pt")
    full_saved_cfg = dict(cd)
    full_saved_cfg.update(mcfg.__dict__)
    full_saved_cfg.update(mech)
    torch.save({
        "step": steps,
        "config": full_saved_cfg,
        "model_config": mcfg.__dict__,
        "model": model.state_dict(),
        "final_loss": last_loss,
    }, ckpt_path)
    
    with open(os.path.join(run_dir, "resolved_config.json"), "w", encoding="utf-8") as fp:
        json.dump(full_saved_cfg, fp, indent=2, ensure_ascii=False)
        
    print(f"  ✓ 训练完成，模型已保存: {ckpt_path} (耗时: {train_duration:.1f}s)")
    
    # 立即对该实验执行 40 题严格评测
    print(f"  🧪 正在执行 40 题零容错严格评测...")
    score, test_details = verify_model_40_questions(
        model, device=device,
        answer_order=mech["answer_order"],
        require_tags=mech["use_ans_tags"]
    )
    print(f"  📊 评测得分: {score}/40 ({score/40*100:.1f}%) | 标签: {mech['use_ans_tags']} | 顺序: {mech['answer_order']}")
    
    all_experiment_reports.append({
        "config": cfg_file,
        "title": title,
        "score": score,
        "duration": train_duration,
        "vocab_size": mech["vocab_size"],
        "details": test_details,
        "cfg_dict": cd,
        "ckpt_path": ckpt_path,
    })
    
    # 显存清理与垃圾回收，确保每轮实验环境纯净
    del model, opt, x, y, logits, loss
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("\n" + "🎉" * 32)
print(f"✅ 全部 {len(target_configs)} 项实验顺序训练与 40 题评测执行完毕！")
print("🎉" * 32)


### Step 5: Benchmark Scorecard & Breakdown View
Display comprehensive scoring details across all evaluated models.


In [ ]:
from tabulate import tabulate

for rep in all_experiment_reports:
    print()
    print("#" * 80)
    print(f"📋 Experiment Report: {rep['title']} ({rep['config']})")
    print(f"得分: {rep['score']}/40 ({rep['score']/40*100:.1f}%) | 词表: {rep['vocab_size']} | 耗时: {rep['duration']:.1f}s")
    print("#" * 80)
    
    table_data = []
    for d in rep['details']:
        status_icon = "🟢 PASS (1分)" if d['pass'] else "🔴 FAIL (0分)"
        pred_display = str(d['pred']) if d['pred'] is not None else "格式错/空"
        table_data.append([d['qid'], d['expr'], d['target'], pred_display, status_icon, d['desc']])
        
    print(tabulate(table_data, headers=["Question", "算式", "真值", "Model Output", "判定结果", "题型特点"], tablefmt="grid"))


### Step 6: Generate Master Mechanistic Conclusions Report
Synthesizes empirical observations and mechanistic causal explanations into `EXPERIMENT_CONCLUSIONS_REPORT.md`.


In [ ]:
report_path = "EXPERIMENT_CONCLUSIONS_REPORT.md"

# Dynamic metrics summary from actual execution
report_lines = [
    "# 📑 SimpleAI 本次 Colab 真实运行评测报告 (Actual Run Report)",
    "",
    f"> **评测时间**：{time.strftime('%Y-%m-%d %H:%M:%S')}",
    f"> **执行硬件**：{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}",
    "> **数据真实性说明**：本报告所有指标均由本次 Colab 运行 40 道测试题零容错判定引擎现场评测得出，绝无任何人工预设或硬编码数据。",
    "",
    "---",
    "",
    "## 🏆 一、本次实际运行得分总览",
    "",
    "| 实验配置编号 | 实验标题 | 词表 | 40题总得分 | 得分率 | 耗时 (s) | 状态 |",
    "|---|---|:---:|:---:|:---:|:---:|:---:|"
]

if not all_experiment_reports:
    report_lines.append("| — | 本次未运行任何实验 | — | 0/40 | 0.0% | 0s | 未执行 |")
else:
    for rep in all_experiment_reports:
        status = "🟢 优秀 (>=80%)" if rep["score"] >= 32 else ("🟡 及格 (>=60%)" if rep["score"] >= 24 else "🔴 待攻关 (<60%)")
        report_lines.append(f"| `{rep['config']}` | {rep['title']} | {rep['vocab_size']} | **{rep['score']}/40** | {rep['score']/40*100:.1f}% | {rep['duration']:.1f}s | {status} |")

report_lines.extend([
    "",
    "---",
    "",
    "## 🔬 二、各实验 40 题真实能力分项实测明细",
    ""
])

for rep in all_experiment_reports:
    details = rep["details"]
    
    # 真实统计分项得分
    add1_pass = sum(1 for d in details if d["qid"] in ["Q01","Q02","Q03","Q04","Q05"] and d["pass"])
    add2_pass = sum(1 for d in details if d["qid"] in ["Q06","Q07","Q08","Q09","Q10"] and d["pass"])
    add3_pass = sum(1 for d in details if d["qid"] in ["Q11","Q12","Q13","Q14","Q15"] and d["pass"])
    add4_pass = sum(1 for d in details if d["qid"] in ["Q16","Q17","Q18","Q19","Q20"] and d["pass"])
    sub1_pass = sum(1 for d in details if d["qid"] in ["Q21","Q22","Q23","Q24","Q25"] and d["pass"])
    sub2_pass = sum(1 for d in details if d["qid"] in ["Q26","Q27","Q28","Q29","Q30"] and d["pass"])
    sub3_pass = sum(1 for d in details if d["qid"] in ["Q31","Q32","Q33","Q34","Q35"] and d["pass"])
    sub4_pass = sum(1 for d in details if d["qid"] in ["Q36","Q37","Q38","Q39","Q40"] and d["pass"])
    
    # 关键机制题检测
    q14_pass = next((d["pass"] for d in details if d["qid"] == "Q14"), False) # 999+1
    q18_pass = next((d["pass"] for d in details if d["qid"] == "Q18"), False) # 9999+1
    q36_pass = next((d["pass"] for d in details if d["qid"] == "Q36"), False) # 1000-1
    
    report_lines.extend([
        f"### 实验: `{rep['config']}` — {rep['title']}",
        f"* **实测总得分**: **{rep['score']} / 40** ({rep['score']/40*100:.1f}%)",
        f"* **加法分项掌握率**:",
        f"  * 1位加法 (Add1): {add1_pass}/5 ({add1_pass*20}%)",
        f"  * 2位加法 (Add2): {add2_pass}/5 ({add2_pass*20}%)",
        f"  * 3位加法 (Add3): {add3_pass}/5 ({add3_pass*20}%)",
        f"  * 4位加法 (Add4): {add4_pass}/5 ({add4_pass*20}%)",
        f"* **减法分项掌握率**:",
        f"  * 1位减法 (Sub1): {sub1_pass}/5 ({sub1_pass*20}%)",
        f"  * 2位减法 (Sub2): {sub2_pass}/5 ({sub2_pass*20}%)",
        f"  * 3位减法 (Sub3): {sub3_pass}/5 ({sub3_pass*20}%)",
        f"  * 4位减法 (Sub4): {sub4_pass}/5 ({sub4_pass*20}%)",
        f"* **极端进位/退位雪崩探针真值**:",
        f"  * `Q14 (999+1)`: {'🟢 PASS' if q14_pass else '🔴 FAIL'}",
        f"  * `Q18 (9999+1)`: {'🟢 PASS' if q18_pass else '🔴 FAIL'}",
        f"  * `Q36 (1000-1)`: {'🟢 PASS' if q36_pass else '🔴 FAIL'}",
        f"* **机制归因初判**: {'【连续进位掌握良好】多位进位雪崩测试均已攻破' if (q14_pass and q18_pass) else '【进位链路仍存在瓶颈】高位或雪崩进位题仍有失分，注意力在长程反向寻址上仍有衰减'}",
        ""
    ])

report_lines.extend([
    "---",
    "*本报告由 Google Colab 现场实测生成 | 严禁伪造数据*"
])

with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

print(f"✅ 基于实测数据的真实评测报告已生成: {report_path}")


### Step 7: Archive All Artifacts to Google Drive
Sync model checkpoints, logs, reports, and Excel workbooks to persistent Google Drive storage.


In [ ]:
import pandas as pd
from google.colab import files

print("=" * 65)
print("💾 Archiving all experimental artifacts to persistent storage...")
print("=" * 65)

# 1. 构造 40 题全量得分明细表格
flat_rows = []
for rep in all_experiment_reports:
    for d in rep["details"]:
        flat_rows.append({
            "实验配置": rep["config"],
            "实验标题": rep["title"],
            "总得分": rep["score"],
            "词表": rep["vocab_size"],
            "耗时(s)": f"{rep['duration']:.1f}",
            "题号": d["qid"],
            "算式": d["expr"],
            "标准真值": d["target"],
            "模型预测": d["pred"] if d["pred"] is not None else "None(未闭合/格式错)",
            "判定结果": "PASS" if d["pass"] else "FAIL",
            "题型归类": d["desc"]
        })

df_scorecard = pd.DataFrame(flat_rows)
scorecard_csv = "evaluation_40_questions_scorecard.csv"
df_scorecard.to_csv(scorecard_csv, index=False, encoding="utf-8-sig")
print(f"✓ 40 题逐题得分明细大表已生成: {scorecard_csv} (共 {len(df_scorecard)} 行)")

# 2. 如果挂载了 Google Drive，自动全量同步
if DRIVE_DIR:
    shutil.copy("EXPERIMENT_CONCLUSIONS_REPORT.md", os.path.join(DRIVE_DIR, "EXPERIMENT_CONCLUSIONS_REPORT.md"))
    shutil.copy(scorecard_csv, os.path.join(DRIVE_DIR, scorecard_csv))
    print(f"✓ 报告与数据表已同步至 Google Drive: {DRIVE_DIR}")
    
    if os.path.exists("runs"):
        runs_dest = os.path.join(DRIVE_DIR, "runs")
        os.makedirs(runs_dest, exist_ok=True)
        os.system(f"cp -ru runs/* {runs_dest}/ 2>/dev/null || true")
        print(f"✓ Checkpoints 与 Runs 训练日志已备份至: {runs_dest}")
    print()
    print(f"🎉 恭喜！全部实验产物已安全归档至 Google Drive 目录: {DRIVE_DIR}")
else:
    print("ℹ️ 未挂载 Drive，产物保存在 Colab 本地。")

# 3. 双重保险：自动打包全部权重与表格为 zip 并提供浏览器直接下载
print("\n📦 正在打包全部模型权重与评测大表为 experiment_artifacts.zip...")
os.system("zip -q -r experiment_artifacts.zip evaluation_40_questions_scorecard.csv EXPERIMENT_CONCLUSIONS_REPORT.md runs/ 2>/dev/null || true")
if os.path.exists("experiment_artifacts.zip"):
    files.download("experiment_artifacts.zip")
    print("✓ 浏览器下载任务已触发！")


### Step 8: Standalone Evaluation & Checkpoint Re-Scoring Engine
Scans all existing checkpoints in `runs/`, runs benchmark evaluations, and syncs scorecards.


In [ ]:
import os, glob, time, shutil, json
import torch
import pandas as pd
from tabulate import tabulate
from google.colab import files
from additive_rand_transformer.model import TinyGPT, TinyGPTConfig
from additive_rand_transformer.data import extract_answer
from additive_rand_transformer.batch_train import verify_model_40_questions, parse_exp_mechanism

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔍 Launching standalone evaluation engine, device: {device}")

# 1. 扫描 runs/ 目录下所有已保存的 checkpoint_final.pt（按生成时间排序）
all_ckpts = sorted(glob.glob("runs/**/checkpoint_final.pt", recursive=True), key=os.path.getmtime)
print(f"📦 共检索到 {len(all_ckpts)} 个已保存的 Checkpoint 文件:")
for p in all_ckpts:
    print(f"  - {p}")

rescued_reports = []

for ckpt_path in all_ckpts:
    run_dir = os.path.dirname(ckpt_path)
    run_name = os.path.basename(run_dir)
    print()
    print("=" * 70)
    print(f"🚀 正在评测已存权重: {ckpt_path}")
    print("=" * 70)
    
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    raw_cfg = ck.get("config", {})
    model_cfg = TinyGPTConfig(**{k: v for k, v in ck.get("model_config", raw_cfg).items() if hasattr(TinyGPTConfig, k)})
    eval_model = TinyGPT(model_cfg).to(device)
    eval_model.load_state_dict(ck["model"])
    
    # 智能解析实验机制（自适应 LSD / <ANS> 标签 / Looped-UT 等）
    mech = parse_exp_mechanism(run_name, raw_cfg)
    answer_order = mech["answer_order"]
    use_ans_tags = mech["use_ans_tags"]
    
    score, test_details = verify_model_40_questions(eval_model, device=device,
                                                    answer_order=answer_order,
                                                    require_tags=use_ans_tags)
    
    print(f"✅ 评测完毕! 40题实测得分: {score}/40 ({score/40*100:.1f}%) | 标签模式: {use_ans_tags} | 答案顺序: {answer_order}")
    
    rescued_reports.append({
        "ckpt_path": ckpt_path,
        "run_name": run_name,
        "score": score,
        "vocab_size": mech["vocab_size"],
        "answer_order": answer_order,
        "use_ans_tags": use_ans_tags,
        "details": test_details
    })

# 2. 构造 40 题逐题明细大表
flat_rows = []
for rep in rescued_reports:
    for d in rep["details"]:
        flat_rows.append({
            "运行目录": rep["run_name"],
            "权重路径": rep["ckpt_path"],
            "40题总得分": rep["score"],
            "得分率": f"{rep['score']/40*100:.1f}%",
            "题号": d["qid"],
            "算式": d["expr"],
            "标准真值": d["target"],
            "模型预测": d["pred"] if d["pred"] is not None else "None(空/未闭合)",
            "判定": "PASS" if d["pass"] else "FAIL",
            "题型特点": d["desc"]
        })

df_rescued = pd.DataFrame(flat_rows)
scorecard_path = "evaluation_40_questions_scorecard.csv"
df_rescued.to_csv(scorecard_path, index=False, encoding="utf-8-sig")
print()
print(f"📊 40 题逐题得分明细大表已更新: {scorecard_path} (共 {len(df_rescued)} 行)")

# 3. 自动同步至 Google Drive 与浏览器打包下载
drive_dest = DRIVE_DIR if 'DRIVE_DIR' in globals() and DRIVE_DIR else '/content/drive/MyDrive/SimpleAI_Experiments'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_dest, exist_ok=True)
    shutil.copy(scorecard_path, os.path.join(drive_dest, scorecard_path))
    os.system(f"cp -ru runs/ {drive_dest}/runs/ 2>/dev/null || true")
    print(f"🎉 全部已存模型权重与 40 题实测得分明细已成功备份至 Google Drive: {drive_dest}")

print("\n📦 正在打包全部模型权重与评测大表为 experiment_artifacts.zip...")
os.system("zip -q -r experiment_artifacts.zip evaluation_40_questions_scorecard.csv runs/ 2>/dev/null || true")
if os.path.exists("experiment_artifacts.zip"):
    files.download("experiment_artifacts.zip")
    print("✓ 浏览器下载任务已触发！")
